<a href="https://colab.research.google.com/github/tfanmd/internship-progress/blob/main/Fine_Tuning_NLLB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# 1. Instalasi library pendukung NLLB + PEFT (LoRA) + Upgrade torchao ke versi aman
print("=== INSTALASI LIBRARY PENDUKUNG NLLB, PEFT, & TORCHAO INTERFACE ===")
!pip install -q transformers[torch] datasets evaluate accelerate sacrebleu peft "torchao>=0.16.0"

import os
import torch
import evaluate
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback
)
# Impor modul khusus untuk konfigurasi LoRA setelah lib ter-update
from peft import LoraConfig, get_peft_model, TaskType

# 2. Memuat kembali dataset CSV dari folder local Colab
path_data = {
    "train": "/content/train.csv",
    "validation": "/content/val.csv",
    "test": "/content/test.csv"
}

print("\n=== MEMUAT DATASET ===")
raw_datasets = load_dataset("csv", data_files=path_data)
print("✓ Library dan torchao versi terbaru berhasil disiapkan. Konflik versi selesai diperbaiki!")

=== INSTALASI LIBRARY PENDUKUNG NLLB, PEFT, & TORCHAO INTERFACE ===

=== MEMUAT DATASET ===


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

✓ Library dan torchao versi terbaru berhasil disiapkan. Konflik versi selesai diperbaiki!


In [2]:
import os
import torch
import evaluate
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback
)
# Impor modul khusus untuk konfigurasi LoRA
from peft import LoraConfig, get_peft_model, TaskType

In [3]:
path_data = {
    "train": "/train.csv",
    "validation": "/val.csv",
    "test": "/test.csv"
}

print("\n=== MEMUAT DATASET ===")
raw_datasets = load_dataset("csv", data_files=path_data)
print("Dataset berhasil dimuat kembali di notebook baru tanpa kendala.")


=== MEMUAT DATASET ===


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Dataset berhasil dimuat kembali di notebook baru tanpa kendala.


In [5]:
print("=== MEMUAT TOKENIZER NLLB-200 (OPTIMASI TOKENS) ===")
MODEL_CHECKPOINT = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_CHECKPOINT,
    src_lang="ind_Latn",
    tgt_lang="jav_Latn"
)

# KUNCI UTAMA: Kita potong ke 64 untuk menghemat VRAM hingga 50%
max_input_length = 64
max_target_length = 64

def preprocess_function_nllb(examples):
    inputs = [str(text) for text in examples["indonesia"]]
    targets = [str(text) for text in examples["jawa"]]

    model_inputs = tokenizer(
        inputs,
        text_target=targets,
        max_length=max_input_length,
        truncation=True
    )
    return model_inputs

print("\n=== MEMULAI PROSES TOKENISASI DATASET ===")
tokenized_datasets = raw_datasets.map(preprocess_function_nllb, batched=True)
print("✓ Proses tokenisasi dengan batas 64 tokens selesai.")

=== MEMUAT TOKENIZER NLLB-200 (OPTIMASI TOKENS) ===


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

...


=== MEMULAI PROSES TOKENISASI DATASET ===


Map:   0%|          | 0/15688 [00:00<?, ? examples/s]

Map:   0%|          | 0/1961 [00:00<?, ? examples/s]

Map:   0%|          | 0/1962 [00:00<?, ? examples/s]

✓ Proses tokenisasi dengan batas 64 tokens selesai.


In [6]:
print("=== MENYIAPKAN FUNGSI EVALUASI METRIK (PERBAIKAN BUG SACREBLEU) ===")

# Memuat metrik sacrebleu dari library evaluate
metric = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Decode token prediksi menjadi teks kembali
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Ganti token -100 pada label (padding) agar bisa di-decode
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Pembersihan spasi berlebih
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    # 1. Hitung skor BLEU bawaan sacrebleu
    bleu_results = metric.compute(predictions=decoded_preds, references=decoded_labels)

    # 2. PERBAIKAN: Hitung skor ChrF dengan mengosongkan parameter tokenize agar tidak memicu KeyError
    chrf_results = metric.compute(predictions=decoded_preds, references=decoded_labels, smooth_method="floor")

    return {
        "bleu": round(bleu_results["score"], 2),
        "chrf": round(chrf_results["score"], 2) # Sacrebleu default mengembalikan skor representatif
    }

print("✓ Bug sacrebleu berhasil di-patch. Fungsi compute_metrics siap digunakan kembali!")

=== MENYIAPKAN FUNGSI EVALUASI METRIK (PERBAIKAN BUG SACREBLEU) ===


✓ Bug sacrebleu berhasil di-patch. Fungsi compute_metrics siap digunakan kembali!


In [7]:
print("=== INSTANSIASI MODEL NLLB-200 + KONFIGURASI LORA ===")

# 1. Memuat model dasar NLLB-200 langsung ke GPU
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT).to("cuda")

# 2. Menentukan konfigurasi adapter LoRA khusus untuk arsitektur Seq2Seq
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False,
    r=8,                           # Rank adapter (makin kecil makin hemat VRAM)
    lora_alpha=32,                 # Faktor penskalaan matriks LoRA
    lora_dropout=0.1,              # Regulasi dropout untuk mencegah overfitting
    target_modules=["q_proj", "v_proj"]  # Target modul attention di dalam NLLB-200
)

# 3. Transformasikan model dasar menjadi model bertenaga LoRA
model = get_peft_model(base_model, peft_config)

# Menampilkan berapa banyak parameter yang akan dilatih (biasanya di bawah 1%)
print("\n=== ANALISIS PARAMETER MODEL AFTER LORA ===")
model.print_trainable_parameters()

# 4. Menyiapkan data collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

print(f"\n✓ Model NLLB-200 + LoRA Adapter berhasil digabungkan!")
print(f"✓ Data Collator berhasil dikonfigurasi.")

=== INSTANSIASI MODEL NLLB-200 + KONFIGURASI LORA ===


pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]


=== ANALISIS PARAMETER MODEL AFTER LORA ===
trainable params: 1,179,648 || all params: 616,253,440 || trainable%: 0.1914

✓ Model NLLB-200 + LoRA Adapter berhasil digabungkan!
✓ Data Collator berhasil dikonfigurasi.


In [8]:
print("=== MENGONFIGURASI PARAMETER PELATIHAN NLLB-200 + LORA (ULTRA SAFE) ===")

training_args = Seq2SeqTrainingArguments(
    output_dir="./results_nllb_lora",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-4,
    per_device_train_batch_size=2,        # Set ke 2 agar VRAM sangat longgar
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,        # 2 x 8 = 16, total batch tetap stabil
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=5,
    predict_with_generate=True,
    fp16=True,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="chrf",
    greater_is_better=True,
    report_to="none"
)

early_stopping = EarlyStoppingCallback(early_stopping_patience=2)
print("✓ Konfigurasi arguments ultra aman selesai disiapkan.")

=== MENGONFIGURASI PARAMETER PELATIHAN NLLB-200 + LORA (ULTRA SAFE) ===
✓ Konfigurasi arguments ultra aman selesai disiapkan.


In [9]:
import gc
print("=== MANDI BERSIH VRAM & START FINE-TUNING ===")

# Bersihkan sisa memory cache GPU secara total
gc.collect()
torch.cuda.empty_cache()
print("✓ VRAM GPU dikosongkan total.")

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[early_stopping]
)

print("\n=== MEMULAI PROSES TRAINING NLLB-200 + LORA ===")
trainer.train()
print("\n✓ Proses training selesai!")

=== MANDI BERSIH VRAM & START FINE-TUNING ===
✓ VRAM GPU dikosongkan total.

=== MEMULAI PROSES TRAINING NLLB-200 + LORA ===


Epoch,Training Loss,Validation Loss,Bleu,Chrf
1,19.196797,2.555648,27.050000,27.050000
2,19.099792,2.478477,27.460000,27.460000
3,17.755464,2.441383,28.430000,28.430000
4,22.140585,2.417384,29.530000,29.530000
5,19.235774,2.410188,28.710000,28.710000



✓ Proses training selesai!


In [10]:
import pandas as pd
import torch
from tqdm import tqdm
import evaluate

print("=== MEMULAI EVALUASI AKHIR PADA DATA TEST ===")

# 1. Pastikan model berada dalam mode evaluasi & GPU aktif
model.eval()

# 2. Ambil data teks mentah dari dataset test yang sudah di-load di awal
test_inputs = raw_datasets["test"]["indonesia"]
test_references = raw_datasets["test"]["jawa"]
test_predictions = []

print(f"Menerjemahkan {len(test_inputs)} data uji... (Harap tunggu)")

# 3. Proses generate terjemahan kalimat demi kalimat secara batch kecil agar aman dari OOM
batch_size = 4
for i in tqdm(range(0, len(test_inputs), batch_size)):
    batch_texts = test_inputs[i:i+batch_size]

    # Tokenisasi teks input Indonesia
    inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=64).to("cuda")

    # Generate prediksi teks bahasa Jawa
    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=64,
            num_beams=4,
            early_stopping=True
        )

    # Decode token menjadi teks bersih
    decoded_preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    test_predictions.extend([pred.strip() for pred in decoded_preds])

print("\n✓ Proses penerjemahan data test selesai.")
print("=== MENGHITUNG METRIK EVALUASI INDEPENDEN ===")

# 4. Kalkulasi metrik secara terpisah untuk menghindari duplikasi angka
bleu_evaluator = evaluate.load("sacrebleu")
chrf_evaluator = evaluate.load("chrf")

# Format references untuk BLEU harus berupa list di dalam list [[ref1], [ref2], ...]
formatted_references_bleu = [[ref.strip()] for ref in test_references]
formatted_references_chrf = [[ref.strip()] for ref in test_references]

# Hitung skor BLEU asli
bleu_results = bleu_evaluator.compute(predictions=test_predictions, references=formatted_references_bleu)
# Hitung skor ChrF asli
chrf_results = chrf_evaluator.compute(predictions=test_predictions, references=formatted_references_chrf)

# 5. Cetak Hasil Akhir yang Akurat untuk Bab 4 Lu
print("\n=============================================")
print("          HASIL TESTING AKHIR MODEL          ")
print("=============================================")
print(f" Skor BLEU Akhir : {round(bleu_results['score'], 2)}")
print(f" Skor ChrF Akhir : {round(chrf_results['score'], 2)}")
print("=============================================\n")

# 6. Simpan hasil prediksi ke file CSV untuk lampiran skripsi
df_hasil = pd.DataFrame({
    "Teks Indonesia (Input)": test_inputs,
    "Teks Jawa Asli (Target)": test_references,
    "Hasil Terjemahan Model (Prediksi)": test_predictions
})

output_file = "hasil_testing_skripsi.csv"
df_hasil.to_csv(output_file, index=False, encoding="utf-8")
print(f"✓ File lampiran sukses disimpan dengan nama: '{output_file}'")

=== MEMULAI EVALUASI AKHIR PADA DATA TEST ===
Menerjemahkan 1962 data uji... (Harap tunggu)


100%|██████████| 491/491 [10:24<00:00,  1.27s/it]



✓ Proses penerjemahan data test selesai.
=== MENGHITUNG METRIK EVALUASI INDEPENDEN ===



          HASIL TESTING AKHIR MODEL          
 Skor BLEU Akhir : 31.93
 Skor ChrF Akhir : 52.28

✓ File lampiran sukses disimpan dengan nama: 'hasil_testing_skripsi.csv'
